# Agent 2 — Step B: Feature Engineering + Train/Test Split  (v3 — with lag features)

**Input:** `master_prices.csv` (1,853 rows from Step A)

**Output:** `train.csv`, `test.csv` with engineered features ready for XGBoost.

**What changed from v2:**
- ➕ Added **lag features** per (millet, state, district): `lag_1m`, `lag_12m`, `rolling_3m_mean`. These are the strongest predictors in commodity price forecasting (every published baseline uses them).
- ⚠️ Rows without enough history are dropped. Expect ~1,200–1,400 rows after pruning (down from 1,853).
- 📈 Expected MAPE: 9–11% (down from 13.5% in v2).

**Still no grade expansion** — grade adjustment is applied in `price_agent.py` post-prediction.

**This step does:**
1. Sort by (millet, state, district, year, month)
2. Compute lag_1m, lag_12m, rolling_3m_mean per group
3. Drop rows with NaN in any lag column
4. Cyclic month features (sin/cos)
5. Encode categoricals (millet, state, district, season)
6. Time-based split: train on 2021–2024, test on 2025–2026

## Cell 1 — Load master_prices.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd
import numpy as np

DRIVE = "/content/drive/MyDrive/MilletSaarthi"
df = pd.read_csv(f"{DRIVE}/master_prices.csv")
print("Loaded rows:", len(df))
print(df.head())

## Cell 2 — Sort and add lag features

Lags are computed per (millet, state, district) so we don't leak across districts. We use:
- `lag_1m` — previous month's price for the same (millet, district)
- `lag_12m` — same month last year
- `rolling_3m_mean` — average of last 3 months (computed *before* the current row to avoid leakage)

Rows where any lag is NaN (e.g. earliest months for each district) are dropped.

In [ ]:
# Sort so groupby+shift gives correct chronological lags
df = df.sort_values(["millet", "state", "district", "year", "month"]).reset_index(drop=True)

g = df.groupby(["millet", "state", "district"])
df["lag_1m"]  = g["modal_price"].shift(1)
df["lag_12m"] = g["modal_price"].shift(12)
# Rolling mean of past 3 months — shift first, then roll, INSIDE each group
# (using transform so rolling never crosses district boundaries)
df["rolling_3m_mean"] = g["modal_price"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=3).mean()
)

before = len(df)
df = df.dropna(subset=["lag_1m", "lag_12m", "rolling_3m_mean"]).reset_index(drop=True)
after = len(df)
print(f"Rows before lag pruning: {before}")
print(f"Rows after  lag pruning: {after}  (dropped {before - after})")
print("\nSample with lag features:")
print(df[["millet","state","district","year","month","modal_price","lag_1m","lag_12m","rolling_3m_mean"]].head(10))


## Cell 3 — Cyclic month features + categorical encoding

In [ ]:
# Cyclic month features capture seasonality (Jan close to Dec)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Season tag (useful for explainability later)
def season(m):
    if m in (6, 7, 8, 9): return "kharif"
    if m in (10, 11, 12, 1, 2, 3): return "rabi"
    return "summer"
df["season"] = df["month"].apply(season)

# Label-encode categoricals (XGBoost handles integer categories well).
# Note: 'grade' is intentionally NOT encoded — grade is applied in price_agent.py.
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ["millet", "state", "district", "season"]:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique values")

# Save encoders for inference later
import pickle
with open(f"{DRIVE}/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
print("\n✅ Saved encoders.pkl  (millet, state, district, season — NO grade)")

## Cell 4 — Time-based train/test split

Train on **2021–2024**, test on **2025–2026** — simulates real forward prediction.

In [ ]:
FEATURES = [
    "millet_enc", "state_enc", "district_enc", "season_enc",
    "year", "month", "month_sin", "month_cos",
    "lag_1m", "lag_12m", "rolling_3m_mean",
]
TARGET = "modal_price"

train = df[df["year"] <= 2024].copy()
test  = df[df["year"] >= 2025].copy()

print(f"Train rows: {len(train)}  (years {train['year'].min()}–{train['year'].max()})")
print(f"Test  rows: {len(test)}  (years {test['year'].min()}–{test['year'].max()})")

train.to_csv(f"{DRIVE}/train.csv", index=False)
test.to_csv(f"{DRIVE}/test.csv", index=False)

# Save feature list for Step C
with open(f"{DRIVE}/features.txt", "w") as f:
    f.write(",".join(FEATURES))

print("\n✅ Saved train.csv, test.csv, features.txt")
print("\nFeatures used:", FEATURES)
print("\nTrain sample:")
print(train[FEATURES + [TARGET]].head())